# Etapa 6: Feature Engineering
---

In [2]:
# Imports
import sys
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [3]:
# Ruta raíz del proyecto (cwd = donde se encuentra el notebook; .parent = ruta padre, eso da la ruta raíz)
PROJECT_ROOT = Path.cwd().parent

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "bank_marketing.csv"

df = pd.read_csv(PROCESSED_PATH)

In [4]:
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

---
## Repaso del decisiones


### Codificación

Para codificar las categorías de cada una de las variables predictoras se usará One-Hot Encoding. La razón principal es porque, mientras los modelos de KNN y RL se basen en distancias, al usar otro método como Label Encoding, interpretarían que una categoría está más lejana que otra; con One-Hot Encoding evita esos errores al crear una columna booleana indicando a cual categoría pertenece la instancia, marcandola con un `1` y a las que no pertenece con `0`.

En cuanto a la conocida "maldición de la dimensionalidad" en machine learning, no es un problema en este caso, ya codificadas el total de columnas sería aproximadament 50. Contemplando que se cuenta con ~45,000 instancias, tener 50 columnas no será un problema en el entrenamiento.

- La variable target `y` se excluye de dicha codificación, simplemente se codifica como 0 si `no` y 1 si `yes`.

- Se utiliza `handle_unknown="ignore"` para que el pipeline pueda procesar categorías que no hayan aparecido durante el entrenamiento, favoreciendo su reutilización en producción.

### Escalado

El escalado aplica unicamente para el modelo de KNN y el de Regresión Logística. Esto debido a que ambos modelos requieren escalado a diferencia de los árboles. Por un lado porque KNN funciona por distancias, por ende, si se mantienen rangos muy distintos como es el caso de `balance` y `age`, `age` quedará casi completamente dominado por la diferencia de `balance`; mientras que Regresión Logistica lo requiere porque se entrena con regularización y esta va a penalizar de forma desigual a las escalas grandes como en el caso de `balance`. Por otro lado, en Árbol de decisión y Random Forest, las variables numéricas se mantienen en su escala original, sin aplicar escalado ni transformaciones por asimetría, ya que los modelos basados en árboles no dependen de la escala de las variables ni requieren que estas presenten distribuciones normales.

Se considera ideal usar el escalado RobustScaler. La decisión se justifica en base a que los otros métodos más conocidos son bastante sensibles a outliers, y durante el diagnóstico se identificaron casos de outliers y de asimetría en algunas de las variables numéricas que estas se consideraron casos posibles del negocio.

### Balanceo de clases

El dataset presenta un desbalance considerable en la variable objetivo (88.3% `no` / 11.7% `yes`), identificado durante el EDA. Para tratarlo, se decidió una estrategia distinta según el modelo, ya que no todos soportan las mismas técnicas.

- Para Árbol de Decisión, Random Forest y Regresión Logística se utilizará `class_weight="balanced"`, disponible de forma nativa en los tres. Esta opción no modifica los datos, sino que penaliza más los errores en la clase minoritaria durante el entrenamiento, sin necesidad de generar registros adicionales. Su efectividad será posteriormente evaluada mediante los experimentos registrados en MLflow, utilizando métricas como Precision, Recall, F1 y AUC.

- Para KNN se utilizará SMOTE, ya que este modelo no cuenta con un parámetro equivalente a `class_weight`. SMOTE genera registros sintéticos de la clase minoritaria interpolando entre vecinos existentes, aplicado únicamente sobre el conjunto de entrenamiento y después del split, para evitar fuga de información hacia el conjunto de prueba.

Se es consciente de que esto puede generar columnas categóricas con valores intermedios no interpretables directamente (por ejemplo, un valor de 0.35 en una columna binaria de One-Hot Encoding). El impacto real de esta decisión se evaluará de forma empírica durante la etapa de tracking, comparando el desempeño de KNN con y sin SMOTE mediante las métricas correspondientes como Accuracy, F1, Recall, en lugar de asumirlo de antemano.


### Decisión sobre `pdays`

La variable `pdays` se conserva en su forma original, incluyendo el valor `-1`, debido a que este valor representa que el cliente no había sido contactado previamente y, por lo tanto, contiene información útil para el modelo. Tampoco se crea una variable derivada como `contactado_previamente`, ya que esta información puede obtenerse a partir de las variables existentes, principalmente `pdays` y `previous`, por lo que crear una nueva variable podría introducir redundancia sin aportar información adicional relevante.

* Las transformaciones mencionadas anteriormente se encuentran encapsuladas en un pipeline reutilizable para garantizar que el mismo proceso pueda aplicarse durante el entrenamiento y posteriormente en producción.


---

In [5]:
df.sample(5)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
29976,31,entrepreneur,single,unknown,no,301,yes,no,cellular,4,feb,371,1,-1,0,unknown,no
4761,39,blue-collar,married,unknown,no,1096,no,no,unknown,20,may,246,3,-1,0,unknown,no
13119,59,services,divorced,secondary,no,136,no,yes,telephone,8,jul,176,1,-1,0,unknown,no
35171,36,blue-collar,married,secondary,no,89,yes,no,cellular,7,may,62,6,366,2,failure,no
44147,30,student,single,secondary,no,459,no,no,cellular,13,jul,98,2,210,22,success,no


In [6]:
from src.features.build_features import build_pipeline, encode_target, feature_selection

In [7]:
# Separación de las variables predictoras (X) de la variable objetivo (y)
# feature_selection se encarga de eliminar 'duration' y 'y' en X
X = feature_selection(df)
y = encode_target(df["y"])

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

# Verificar las variables que serán utilizadas como predictoras
print("Variables predictoras:")
print(X.columns.tolist())

Dimensiones de X: (45211, 15)
Dimensiones de y: (45211,)
Variables predictoras:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome']


In [8]:
# Para validar que los pipelines funcionan
from src.pipelines.split import split_data

X_train, X_test, y_train, y_test = split_data(X, y)

In [9]:
# Comparamos la distribución de la variable objetivo entre entrenamiento y prueba
# para comprobar que el desbalance de clases se mantiene aproximadamente igual.

print("Distribución en entrenamiento:")
print(y_train.value_counts(normalize=True).round(3))

print("\nDistribución en prueba:")
print(y_test.value_counts(normalize=True).round(3))

Distribución en entrenamiento:
y
0    0.883
1    0.117
Name: proportion, dtype: float64

Distribución en prueba:
y
0    0.883
1    0.117
Name: proportion, dtype: float64


In [10]:
# Pipeline de Decision Tree: 
# Codifica las categóricas y aplica class_weight="balanced" para compensar el desbalance de clases (88.3% no / 11.7% yes).

decision_tree_pipeline = build_pipeline(
    model=DecisionTreeClassifier(random_state=42, class_weight="balanced", max_depth=4, criterion="gini"),
    incluir_escalado=False
)

print("Pipeline de Decision Tree creado correctamente.")

Pipeline de Decision Tree creado correctamente.


In [11]:
# Creamos el pipeline de Random Forest.
# Codifica las categóricas y aplica class_weight="balanced" para compensar el desbalance de clases (88.3% no / 11.7% yes).

random_forest_pipeline = build_pipeline(
    model=RandomForestClassifier(random_state=42, class_weight="balanced", n_estimators=100, max_depth=5, n_jobs=-1),
    incluir_escalado=False
)

print("Pipeline de Random Forest creado correctamente.")

Pipeline de Random Forest creado correctamente.


In [12]:
k = 3  # Valor provisional para validar el pipeline; el valor final se determina en la etapa de tracking (MLflow)

# Pipeline de KNN: 
# Codifica categóricas, escala numéricas con RobustScaler (KNN es sensible a la escala), 
# y aplica SMOTE sobre el train para tratar el desbalance, ya que KNN no soporta class_weight.

knn_pipeline = build_pipeline(
    model=KNeighborsClassifier(n_neighbors=k),
    incluir_escalado=True,
    incluir_smote=True  
)

print("Pipeline de KNN creado correctamente.")

Pipeline de KNN creado correctamente.


In [13]:
# Pipeline de Regresión Logística: 
# Codifica las categóricas con One-Hot y escala las numéricas con RobustScaler (sensible a la escala por la regularización).
# Se usa class_weight="balanced" para compensar el desbalance de clases (88.3% no / 11.7% yes).

rl_pipeline = build_pipeline(
    model=LogisticRegression(random_state=42, class_weight="balanced", max_iter=1000),
    incluir_escalado=True
)

print("Pipeline de Regresión Logística creado correctamente.")

Pipeline de Regresión Logística creado correctamente.
